# UNSDG classification metrics and threshold analysis

This notebook evaluates the **saved ST URL pipeline results** against the 17 ground-truth columns in `backend/csv_data_per_sdg/sdg_*.csv`. It derives exploratory per-SDG cutoffs, reports metrics, and audits the limits of the saved results.

Run top to bottom from the repository root or from `docs/`. It uses only the Python standard library. The adjacent [`classification_results_analysis.md`](classification_results_analysis.md) records the results observed when this analysis was run.

**Important:** the result files contain only scores returned at the existing `>= 0.3` cutoff. A missing class score is treated as `< 0.3` only for a request that has at least one saved score. Requests with no saved scores are excluded from threshold metrics because the exports cannot distinguish an empty/gated response, a no-match result, or a failed request.

In [ ]:
import csv
import math
import re
from collections import Counter, defaultdict
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

ROOT = Path.cwd()
if not (ROOT / "backend" / "csv_data_per_sdg").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "backend" / "csv_data_per_sdg"
RESULT_DIR = DATA_DIR / "res_per_sdg"
SDGS = range(1, 18)
INPUT_REQUIRED = {"name", "repository_link", "description", "target_sdg"} | {
    f"sdg_{n}" for n in SDGS
}
RESULT_REQUIRED = {
    "source_file", "source_row", "repository_link", "prediction", "predicted_sdg"
}

def normalize_repo_url(value):
    parts = urlsplit(value.strip())
    host = parts.netloc.casefold()
    path = parts.path.rstrip("/").removesuffix(".git").casefold()
    return urlunsplit((parts.scheme.casefold(), host, path, "", ""))

def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames:
            raise ValueError(f"Missing CSV header: {path}")
        return reader.fieldnames, list(reader)

input_files = sorted(
    (p for p in DATA_DIR.glob("sdg_*.csv") if re.fullmatch(r"sdg_\d+", p.stem)),
    key=lambda p: int(p.stem.split("_")[1]),
)
if {int(p.stem.split("_")[1]) for p in input_files} != set(SDGS):
    raise ValueError("Expected exactly one input CSV for each SDG 1 through 17")

records = {}
for path in input_files:
    headers, rows = read_csv(path)
    missing = INPUT_REQUIRED - set(headers)
    if missing:
        raise ValueError(f"{path.name} missing columns: {sorted(missing)}")
    for physical_row, row in enumerate(rows, start=2):
        key = (path.name, str(physical_row))
        labels = {}
        for n in SDGS:
            value = row[f"sdg_{n}"].strip()
            if value not in {"0", "1"}:
                raise ValueError(f"Expected binary sdg_{n} in {key}, got {value!r}")
            labels[n] = int(value)
        row["_labels"] = labels
        row["_repo"] = normalize_repo_url(row["repository_link"])
        records[key] = row

repo_labels = {}
repo_runs = defaultdict(list)
for key, row in records.items():
    vector = tuple(row["_labels"][n] for n in SDGS)
    previous = repo_labels.setdefault(row["_repo"], vector)
    if previous != vector:
        raise ValueError(f"Conflicting ground-truth vectors for {row['_repo']}")
    repo_runs[row["_repo"]].append(key)

scores = {}
output_keys = set()
output_row_count = 0
for n in SDGS:
    path = RESULT_DIR / f"sdg_{n}.csv"
    if not path.is_file():
        raise FileNotFoundError(f"Missing result file: {path}")
    headers, rows = read_csv(path)
    missing = RESULT_REQUIRED - set(headers)
    if missing:
        raise ValueError(f"{path.name} missing columns: {sorted(missing)}")
    for row in rows:
        key = (row["source_file"], str(int(row["source_row"])))
        if key not in records:
            raise ValueError(f"Result points to unknown input row: {key}")
        score_key = (key, n)
        if score_key in scores:
            raise ValueError(f"Duplicate saved score: {score_key}")
        score = float(row["prediction"])
        if not math.isfinite(score) or not 0.3 <= score <= 1.0:
            raise ValueError(f"Score outside saved-output range [0.3, 1]: {score}")
        label_match = re.match(r"^\s*SDG\s*(\d+)\b", row["predicted_sdg"], re.I)
        if not label_match or int(label_match.group(1)) != n:
            raise ValueError(f"SDG label/file mismatch in {path.name}: {row['predicted_sdg']!r}")
        scores[score_key] = score
        output_keys.add(key)
        output_row_count += 1

scored_records = [(key, records[key]) for key in records if key in output_keys]
scored_repo_runs = Counter(row["_repo"] for _, row in scored_records)
run_weights = [1.0 / scored_repo_runs[row["_repo"]] for _, row in scored_records]

input_duplicate_urls = sum(len(keys) > 1 for keys in repo_runs.values())
print(f"Input rows: {len(records):,}; unique repository URLs: {len(repo_runs):,}")
print(f"Repository URLs repeated in source files: {input_duplicate_urls:,}; maximum rows for one URL: {max(map(len, repo_runs.values()))}")
print(f"Requests with >=1 saved score: {len(scored_records):,} ({len(scored_records) / len(records):.1%})")
print(f"Unique repositories with scores: {len(scored_repo_runs):,} / {len(repo_runs):,} ({len(scored_repo_runs) / len(repo_runs):.1%})")
print(f"Saved score rows: {output_row_count:,}; possible request/SDG cells: {len(records) * 17:,}")
print(f"Saved score coverage across all possible cells: {output_row_count / (len(records) * 17):.1%}")
print(f"No-output request rows excluded from score metrics: {len(records) - len(scored_records):,}")

## Metric and threshold method

- Evaluation unit: a saved pipeline request for a unique repository. Repeated requests for the same normalized URL are retained to expose score variation, but each repository's total weight is 1 (`1 / its number of score-bearing requests`). Ground-truth vectors are checked for consistency first.
- A class score absent from a score-bearing request is known only to be below `0.3`, because the classifier serializes selected predictions at `>= 0.3`. It is negative at every evaluated threshold (`0.30`–`0.99`). An entire request with no saved scores is not assumed negative and is excluded.
- Per-SDG cutoffs are chosen on the same observed sample by maximizing weighted F1 over a 0.01 grid from `0.30` to `0.99`; ties prefer precision, then the higher cutoff. These are **apparent, in-sample** values, not held-out validation estimates.
- Accuracy is included, but is not the main quality measure because most repository/SDG pairs are negative. Macro and micro precision/recall/F1 and 17-label exact-match accuracy are also reported.

In [ ]:
def confusion_for(goal, threshold):
    tp = fp = fn = tn = 0.0
    for (key, row), weight in zip(scored_records, run_weights):
        truth = row["_labels"][goal]
        score = scores.get((key, goal), float("-inf"))
        predicted = score >= threshold
        if truth and predicted:
            tp += weight
        elif predicted:
            fp += weight
        elif truth:
            fn += weight
        else:
            tn += weight
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn}

def metrics(cm):
    tp, fp, fn, tn = cm["tp"], cm["fp"], cm["fn"], cm["tn"]
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    return {"precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy}

threshold_grid = [i / 100 for i in range(30, 100)]
results_by_goal = {}
for goal in SDGS:
    grid = [(threshold, metrics(confusion_for(goal, threshold))) for threshold in threshold_grid]
    best_threshold, best_metrics = max(
        grid,
        key=lambda item: (item[1]["f1"], item[1]["precision"], item[0]),
    )
    results_by_goal[goal] = {
        "support": len({row["_repo"] for _, row in scored_records if row["_labels"][goal]}),
        "baseline": metrics(confusion_for(goal, 0.30)),
        "baseline_cm": confusion_for(goal, 0.30),
        "threshold": best_threshold,
        "selected": best_metrics,
        "selected_cm": confusion_for(goal, best_threshold),
        "curve": grid,
    }

def fmt(value):
    return f"{value:.3f}"

headers = ["SDG", "Unique positive repos", "P@0.30", "R@0.30", "F1@0.30", "Accuracy@0.30", "Candidate cutoff", "P@candidate", "R@candidate", "F1@candidate", "Accuracy@candidate"]
print("| " + " | ".join(headers) + " |")
print("|" + "|".join(["---"] * len(headers)) + "|")
for goal, result in results_by_goal.items():
    before, after = result["baseline"], result["selected"]
    cells = [str(goal), str(result["support"]), fmt(before["precision"]), fmt(before["recall"]), fmt(before["f1"]), fmt(before["accuracy"]), f"{result['threshold']:.2f}", fmt(after["precision"]), fmt(after["recall"]), fmt(after["f1"]), fmt(after["accuracy"])]
    print("| " + " | ".join(cells) + " |")

In [ ]:
def aggregate_at(thresholds):
    class_metrics = [metrics(confusion_for(goal, thresholds[goal])) for goal in SDGS]
    totals = {name: sum(confusion_for(goal, thresholds[goal])[name] for goal in SDGS) for name in ("tp", "fp", "fn", "tn")}
    micro = metrics(totals)
    exact_weight = 0.0
    for (key, row), weight in zip(scored_records, run_weights):
        exact = all(
            (scores.get((key, goal), float("-inf")) >= thresholds[goal]) == bool(row["_labels"][goal])
            for goal in SDGS
        )
        exact_weight += weight * exact
    macro = {name: sum(item[name] for item in class_metrics) / 17 for name in ("precision", "recall", "f1", "accuracy")}
    return macro, micro, exact_weight / len(scored_repo_runs)

baseline_thresholds = {goal: 0.30 for goal in SDGS}
selected_thresholds = {goal: results_by_goal[goal]["threshold"] for goal in SDGS}
for title, thresholds in (("Current 0.30", baseline_thresholds), ("Per-SDG candidate", selected_thresholds)):
    macro, micro, exact_match = aggregate_at(thresholds)
    print(f"\n{title}")
    print("  Macro P/R/F1/accuracy: " + " / ".join(fmt(macro[name]) for name in ("precision", "recall", "f1", "accuracy")))
    print("  Micro P/R/F1: " + " / ".join(fmt(micro[name]) for name in ("precision", "recall", "f1")))
    print(f"  17-label exact-match accuracy: {exact_match:.1%}")

In [ ]:
def decision_disagreement(thresholds):
    repeated_repos = defaultdict(list)
    for key, row in scored_records:
        repeated_repos[row["_repo"]].append(key)
    comparisons = disagreements = 0
    for repo, keys in repeated_repos.items():
        if len(keys) < 2:
            continue
        for goal in SDGS:
            flags = {scores.get((key, goal), float("-inf")) >= thresholds[goal] for key in keys}
            comparisons += 1
            disagreements += len(flags) > 1
    return disagreements, comparisons

def observed_score_variation():
    repeated_repos = defaultdict(list)
    for key, row in scored_records:
        repeated_repos[row["_repo"]].append(key)
    comparisons = varying = 0
    for keys in repeated_repos.values():
        if len(keys) < 2:
            continue
        for goal in SDGS:
            observed = [scores[(key, goal)] for key in keys if (key, goal) in scores]
            if len(observed) >= 2:
                comparisons += 1
                varying += max(observed) - min(observed) > 1e-5
    return varying, comparisons

varying, comparable = observed_score_variation()
print(f"Repeated repository-SDG pairs with >=2 observed saved scores that vary: {varying}/{comparable} ({varying / comparable:.1%})")

for title, thresholds in (("0.30", baseline_thresholds), ("candidate", selected_thresholds)):
    disagreements, comparisons = decision_disagreement(thresholds)
    print(f"Repeated-repository SDG decisions that change across saved runs ({title}): "
          f"{disagreements}/{comparisons} ({disagreements / comparisons:.1%})")

print("\nHighest-scored false positives at candidate thresholds (up to 3 per goal):")
for goal in SDGS:
    threshold = selected_thresholds[goal]
    errors_by_repo = {}
    for key, row in scored_records:
        score = scores.get((key, goal), float("-inf"))
        if not row["_labels"][goal] and score >= threshold:
            previous = errors_by_repo.get(row["_repo"])
            if previous is None or score > previous[0]:
                errors_by_repo[row["_repo"]] = (score, row["name"])
    errors = list(errors_by_repo.values())
    if errors:
        examples = ", ".join(f"{name} ({score:.3f})" for score, name in sorted(errors, reverse=True)[:3])
        print(f"  SDG {goal}: {examples}")

print("\nHighest-scored missed positives at candidate thresholds (up to 3 per goal):")
for goal in SDGS:
    threshold = selected_thresholds[goal]
    errors_by_repo = {}
    for key, row in scored_records:
        score = scores.get((key, goal), float("-inf"))
        if row["_labels"][goal] and score < threshold:
            previous = errors_by_repo.get(row["_repo"])
            if previous is None or score > previous[0]:
                errors_by_repo[row["_repo"]] = (score, row["name"])
    errors = list(errors_by_repo.values())
    if errors:
        examples = ", ".join(
            f"{name} ({'<0.30; not serialized' if score == float('-inf') else f'{score:.3f}'})"
            for score, name in sorted(errors, reverse=True)[:3]
        )
        print(f"  SDG {goal}: {examples}")

## Interpretation and next steps

Treat the selected cutoffs as **candidates for a new labeled validation set**, not deployment configuration. Because they are chosen and evaluated on the same sample, their F1 values are optimistic. Use repository-grouped train/validation or nested cross-validation for threshold selection, then report performance on untouched repositories.

The CSV exports are left-censored below `0.3` and omit all-zero/empty request records. Consequently this notebook cannot calculate full-range ROC/PR curves, AUROC, average precision, Brier score, probability calibration, failure-adjusted recall, or end-to-end accuracy. Preserve every request outcome and all 17 raw scores (including scores below the display cutoff) in future evaluation exports.

Repeated URLs with differing scores expose run-to-run output variation, but these files do not record the generated summary, raw model scores, or run/model versions needed to locate its cause. Record those inputs and provenance before attributing the variation to a specific stage.